In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import VGG16_Weights

In [3]:
!pip install pyyaml

In [5]:
from ssd import SSD
from utils import normalised_gt_coords,corner_to_center,iou,center_to_corner,decode,encode ,matching



"""targets (tensor): Ground truth boxes and labels for a batch,
                shape: [batch_size,num_objs,5] (last idx is the label)."""

gt_boxes_batch = [
    # Image 1
    [
        (75, 80, 60, 90),
        (220, 70, 50, 40),
        (150, 170, 140, 110),
        (260, 240, 70, 120),
        (45, 230, 80, 60)
    ],

    # Image 2
    [
        (30, 40, 50, 60),
        (200, 120, 80, 90),
        (180, 160, 100, 110),
        (250, 210, 60, 80),
        (90, 200, 70, 50),
        (90, 200, 75, 50)
    ],
]

labels_list = [
    torch.tensor([1, 2, 1, 1, 2]),
    torch.tensor([1, 3, 3, 4, 2, 2]) 
]

# convert to tensors
gt_boxes_list = [torch.tensor(b, dtype=torch.float32) for b in gt_boxes_batch]

# gt=center_to_corner(normalised_gt_coords(gtboxes,300,300))


In [6]:
gt_list=[center_to_corner(normalised_gt_coords(gtboxes,300,300)) for gtboxes in gt_boxes_list]
gt_list

[tensor([[0.1500, 0.1167, 0.3500, 0.4167],
         [0.6500, 0.1667, 0.8167, 0.3000],
         [0.2667, 0.3833, 0.7333, 0.7500],
         [0.7500, 0.6000, 0.9833, 1.0000],
         [0.0167, 0.6667, 0.2833, 0.8667]]),
 tensor([[0.0167, 0.0333, 0.1833, 0.2333],
         [0.5333, 0.2500, 0.8000, 0.5500],
         [0.4333, 0.3500, 0.7667, 0.7167],
         [0.7333, 0.5667, 0.9333, 0.8333],
         [0.1833, 0.5833, 0.4167, 0.7500],
         [0.1750, 0.5833, 0.4250, 0.7500]])]

In [9]:
gt_list[0]

tensor([[0.1500, 0.1167, 0.3500, 0.4167],
        [0.6500, 0.1667, 0.8167, 0.3000],
        [0.2667, 0.3833, 0.7333, 0.7500],
        [0.7500, 0.6000, 0.9833, 1.0000],
        [0.0167, 0.6667, 0.2833, 0.8667]])

In [7]:
X=torch.randn((2,3,300,300))

In [16]:
!pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [kagglehub]/4 [kagglehub]


In [17]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ultralytics/coco128")

print("Path to dataset files:", path)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 6.66M/6.66M [00:00<00:00, 7.00MB/s]

Extracting files...
Path to dataset files: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3


In [19]:
import os

dataset_root = "/home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128"

for root, dirs, files in os.walk(dataset_root):
    print("DIR:", root)
    print("Subdirs:", dirs)
    print("Files:", files[:5])
    print("-" * 50)

DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128
Subdirs: ['labels', 'images']
Files: ['README.txt', 'LICENSE']
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/labels
Subdirs: ['train2017']
Files: []
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/labels/train2017
Subdirs: []
Files: ['000000000294.txt', '000000000036.txt', '000000000471.txt', '000000000491.txt', '000000000315.txt']
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/images
Subdirs: ['train2017']
Files: []
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/images/train2017
Subdirs: []
Files: ['000000000127.jpg', '000000000151.jpg', '000000000073.jpg', '000

In [53]:
import glob

img_dir = dataset_root + "/images/train2017"
lbl_dir = dataset_root + "/labels/train2017"

images = sorted(glob.glob(img_dir + "/*.jpg"))
labels = sorted(glob.glob(lbl_dir + "/*.txt"))

print("Images:", len(images))
print("Labels:", len(labels))
print(images[0])
print(labels[0])

Images: 128
Labels: 128
/home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/images/train2017/000000000009.jpg
/home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/labels/train2017/000000000009.txt


In [47]:
len(images)

128

In [25]:
!pip uninstall -y opencv-python
!pip install opencv-python-headless
import cv2

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 98.6 MB/s  0:00:006m0:00:0100:01


In [44]:
with open(labels[0]) as f:
    gt_box=[]
    label_list=[]
    for line in f:
        label,cx,cy,w,h=map(float,line.split())
        gt_box.append((cx,cy,w,h))
        label_list.append(label)
    gt_box=torch.tensor(gt_box, dtype=torch.float32)
    label_list=torch.tensor(label_list, dtype=torch.int32)
        
 

In [46]:
label_list

tensor([45, 45, 50, 45, 49, 49, 49, 49], dtype=torch.int32)

In [35]:
from PIL import Image
from torchvision import transforms

transform = transforms.ToTensor()

img = Image.open(images[0]).convert("RGB")
img_tensor = transform(img)
img_tensor.shape

torch.Size([3, 480, 640])

In [ ]:
from utils import center_to_corner
class DataSSD300(torch.utils.data.Dataset):
    def __init__(self,img_dir,lbl_dir,gt_normalised : bool = True):
        self.images = sorted(glob.glob(img_dir + "/*.jpg"))
        self.labels = sorted(glob.glob(lbl_dir + "/*.txt"))
        self.transform = transforms.Compose([
            transforms.Resize((300, 300)),
            transforms.ToTensor()            
        ])
        self.gt_normalised=gt_normalised

    def __len__(self):
        return len(self.images)

    def __getitem__(self,idx):

        #images

        img = Image.open(self.images[idx]).convert("RGB")

        if not self.gt_normalised:
            W,H=img.size
        else:
            H,W=1,1
        img_tensor = self.transform(img)

        # labels and gt boxes
        with open(self.labels[idx]) as f:
            gt_box=[]
            label_list=[]
            for line in f:
                label,cx,cy,w,h=map(float,line.split())
                
                gt_box.append((cx/W,cy/H,w/W,h/H))

                label_list.append(label+1)#yolo labels start at 0, my ssd start at 1, O is BG
            gt_box=center_to_corner(torch.tensor(gt_box, dtype=torch.float32)*300)

            label_list=torch.tensor(label_list, dtype=torch.int64)

        return img_tensor,label_list,gt_box

In [137]:
d=[2]
1 or d[-1]

1

In [131]:
233 or 1

233

In [121]:
dataset=DataSSD300(img_dir,lbl_dir)

def collate_ssd(batch):
    images, labels, boxes = zip(*batch)
    images = torch.stack(images, dim=0)
    return images, list(labels), list(boxes)

test_size = 0.15
val_size = 0.15
test_amount, val_amount = int(len(dataset)*test_size),int(len(dataset)*val_size)


# this function will automatically randomly split your dataset but you could also implement the split yourself
train_set, val_set, test_set = torch.utils.data.random_split(dataset, [
            (len(dataset) - (test_amount + val_amount)), 
            test_amount, 
            val_amount
])


train_dataloader = torch.utils.data.DataLoader(
            train_set,
            batch_size=20,
            shuffle=True,
            collate_fn=collate_ssd
)
val_dataloader = torch.utils.data.DataLoader(
            val_set,
            batch_size=20,
            shuffle=True,
            collate_fn=collate_ssd
)
test_dataloader = torch.utils.data.DataLoader(
            test_set,
            batch_size=20,
            shuffle=True,
            collate_fn=collate_ssd
)


In [128]:
for img_tensor,label_list,gt_box in train_dataloader:
    print(img_tensor.shape)
    print(len(label_list))
    print(len(gt_box))
    break

torch.Size([20, 3, 300, 300])
20
20


In [129]:
len(val_dataloader)

1

In [64]:
dataset[1]

(tensor([[[0.0275, 0.0275, 0.0275,  ..., 0.2745, 0.2588, 0.2588],
          [0.0275, 0.0275, 0.0275,  ..., 0.5882, 0.5843, 0.5843],
          [0.0275, 0.0275, 0.0314,  ..., 0.7294, 0.7176, 0.7098],
          ...,
          [0.7608, 0.6549, 0.5686,  ..., 0.5412, 0.5686, 0.4941],
          [0.7294, 0.6275, 0.6039,  ..., 0.4745, 0.5608, 0.5255],
          [0.5608, 0.4784, 0.4784,  ..., 0.4745, 0.5608, 0.5255]],
 
         [[0.0314, 0.0314, 0.0314,  ..., 0.2706, 0.2431, 0.2431],
          [0.0314, 0.0314, 0.0314,  ..., 0.5843, 0.5804, 0.5804],
          [0.0314, 0.0314, 0.0353,  ..., 0.7294, 0.7176, 0.7098],
          ...,
          [0.7333, 0.6353, 0.5412,  ..., 0.4392, 0.4667, 0.3922],
          [0.7098, 0.6157, 0.5843,  ..., 0.3686, 0.4627, 0.4275],
          [0.5490, 0.4667, 0.4667,  ..., 0.3686, 0.4627, 0.4275]],
 
         [[0.0118, 0.0118, 0.0118,  ..., 0.2549, 0.2314, 0.2314],
          [0.0118, 0.0118, 0.0118,  ..., 0.5686, 0.5647, 0.5647],
          [0.0118, 0.0118, 0.0157,  ...,

In [61]:
img=Image.open(images[0]).convert("RGB")
transform = transforms.ToTensor()

transform(img)

tensor([[[0.0078, 0.0078, 0.0078,  ..., 0.5333, 0.5216, 0.5176],
         [0.0078, 0.0078, 0.0078,  ..., 0.5294, 0.5255, 0.5216],
         [0.0078, 0.0078, 0.0039,  ..., 0.5333, 0.5294, 0.5216],
         ...,
         [0.0275, 0.0275, 0.0314,  ..., 0.0157, 0.0078, 0.0275],
         [0.0235, 0.0235, 0.0275,  ..., 0.0392, 0.0392, 0.0353],
         [0.0157, 0.0196, 0.0235,  ..., 0.0549, 0.0667, 0.0784]],

        [[0.0824, 0.0824, 0.0824,  ..., 0.6706, 0.6667, 0.6627],
         [0.0824, 0.0824, 0.0824,  ..., 0.6706, 0.6667, 0.6667],
         [0.0824, 0.0824, 0.0863,  ..., 0.6745, 0.6706, 0.6667],
         ...,
         [0.0000, 0.0000, 0.0078,  ..., 0.0314, 0.0078, 0.0275],
         [0.0000, 0.0000, 0.0000,  ..., 0.0078, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

        [[0.4510, 0.4510, 0.4510,  ..., 0.7490, 0.7373, 0.7333],
         [0.4510, 0.4510, 0.4510,  ..., 0.7569, 0.7529, 0.7412],
         [0.4510, 0.4510, 0.4510,  ..., 0.7765, 0.7725, 0.

In [4]:
vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
vgg[16] = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)
base=nn.ModuleList(vgg[:30])#until 5_3 layer
base




ModuleList(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (17): Conv2d(256, 512, kernel_siz

In [12]:
X=torch.randn((2,3,300,300))

model=SSD(base,21,alpha=0.1)

In [13]:
regressions,classifications=model(X)

In [14]:
classifications.shape

torch.Size([2, 8732, 21])

In [10]:
from multiloss import MultiLoss

In [11]:
m=MultiLoss()
m.forward(0,2,model.anchors,gt_list,labels_list,regressions,classifications)

(tensor(0.9604, grad_fn=<DivBackward0>),
 tensor(15.4805, grad_fn=<DivBackward0>),
 False)

In [261]:
labels_space.shape

torch.Size([2, 8732, 1])

In [262]:
HNM(classifications[0,:,:],labels_space[0,:,:].squeeze(1))

tensor([5479,  302, 5631,  ..., 8591, 8641, 8691])

In [265]:
F.cross_entropy(classifications[0,:,:],labels_space[0,:,:].squeeze(1),reduction="none") 

tensor([2.8834, 2.9954, 3.0398,  ..., 3.0416, 3.0485, 3.0293],
       grad_fn=<NllLossBackward0>)

In [303]:
labels_reshaped=labels_space.permute(1,0,2).squeeze(-1)
classifications_reshaped=classifications.permute(1,2,0)
losses=F.cross_entropy(classifications_reshaped,labels_reshaped,reduction="none")
losses.shape
# negative_indexes=torch.nonzero(labels_reshaped==0,as_tuple=True,)[0]
# positive_indexes=torch.nonzero(labels_reshaped>0,as_tuple=True,)[0]

torch.Size([8732, 2])

In [305]:
labels_space.permute(1,0,2).squeeze(-1)>0

tensor([[False, False],
        [False, False],
        [False, False],
        ...,
        [False, False],
        [False, False],
        [False, False]])

In [ ]:
labels_space.permute(1,0,2).squeeze(-1).shape

In [ ]:
def HNM(classifications_reshaped,labels_reshaped):

    """
    Input: 

    reshaped classifications; shape [N_anhcores(8732),N_classes] 
    reshaped labels; shape [N_anhcores] , one vector containing classes 

    Output:

    all; shape [positive anchors + top negative anchors]

    For more details : https://arxiv.org/pdf/1512.02325, page 6 

    """
    losses=F.cross_entropy(classifications_reshaped,labels_reshaped,reduction="none") 

    negative_indexes=torch.nonzero(labels_reshaped==0,as_tuple=True,)[0]
    positive_indexes=torch.nonzero(labels_reshaped>0,as_tuple=True,)[0]
    nb_positives=positive_indexes.numel()


    _,indx=losses[negative_indexes].sort(descending=True)

    negative_indexes=negative_indexes[indx[:min(nb_positives*4,len(indx))]]
    
    all=torch.cat([negative_indexes, positive_indexes], dim=0)
    return all


# model forward pass 

In [10]:
X=torch.randn((10,3,300,300))

model=SSD(base,21)
regressions,classifications,layers_for_prediction=model.forward(X)

In [11]:
classifications.shape

torch.Size([10, 8732, 21])

In [46]:
regressions.shape



torch.Size([10, 8732, 4])

In [30]:
for el  in layers_for_prediction:
    print(el.shape[2:])



torch.Size([38, 38])
torch.Size([19, 19])
torch.Size([10, 10])
torch.Size([5, 5])
torch.Size([3, 3])
torch.Size([1, 1])
